# Day 10

In [12]:
import fs from 'node:fs';

In [13]:
const input = fs.readFileSync("input.txt", "utf-8");

In [14]:
const sample = `\
[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}`;

I didn't finish part 2 of yesterday's problem yet. Guess the problems are getting harder, or maybe the computational geometry stuff threw me to uncharted teritory. Anyway, better to move on to today's problem.

## Part 1
The first part of today is a graph search. Each row denotes a target state of array binary toggles `#` and `,` (`[.##.]` for example) and a bunch of buttons that trigger some of them. (3) for example is button that triggers the fourth toggles. (2,3) will triger both the third and the fourth toggles. The curly bracket stuff is for the next part so we ignore it for now. 

We are asked to find for each row the minimal number of presses required to turn the machine from the initial off state `[....]` to the target state. We can use BFS for this, have states as nodes and edges for button presses.

In [15]:
const rows = sample.split('\n')

Perhaps today is more fitting for regex.

In [16]:
const button_pat = /\((\d+(?:,\d+)*)\)/gi;
rows[0].match(button_pat);

[ "(3)", "(1,3)", "(2)", "(2,3)", "(0,2)", "(0,1)" ]

In [17]:
const state_pat = /\[[.#]+\]/;
rows[0].match(state_pat);

[
  "[.##.]",
  index: 0,
  input: "[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}",
  groups: undefined
]

In [18]:
const jolt_pat = /{\d+(?:,\d+)*}/;
rows[0].match(jolt_pat);

[
  "{3,5,4,7}",
  index: 39,
  input: "[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}",
  groups: undefined
]

In [19]:
const pat = `(${state_pat.source})(?: ${button_pat.source})+ (${jolt_pat.source})`
console.log(pat);
rows[0].match(pat);

(\[[.#]+\])(?: \((\d+(?:,\d+)*)\))+ ({\d+(?:,\d+)*})


[
  "[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}",
  "[.##.]",
  "0,1",
  "{3,5,4,7}",
  index: 0,
  input: "[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}",
  groups: undefined
]

We are capturing only the last pair... why is that? seems like capture groups are static, that means nesting the capture group inside the + operator causes it to overwrite the previous matches and keep only the last one. Well, in that case, let's just manually split and parse.

In [20]:
const parts = rows[0].split(' ');
const st = parts[0].slice(1,-1).split('').map(e => e === "#");
const buts = parts.slice(1,-1).map(s => s.slice(1,-1).split(',').map(n => parseInt(n)));
const jolt = parts[parts.length-1].slice(1, -1).split(',').map(n => parseInt(n));

({
      st: parts[0].slice(1,-1).split('').map(e => e === "#"),
      buts: parts.slice(1,-1).map(s => s.slice(1,-1).split(',').map(n => parseInt(n))),
      jolt: parts[parts.length-1].slice(1, -1).split(',').map(n => parseInt(n))
})

{
  st: [ false, true, true, false ],
  buts: [ [ 3 ], [ 1, 3 ], [ 2 ], [ 2, 3 ], [ 0, 2 ], [ 0, 1 ] ],
  jolt: [ 3, 5, 4, 7 ]
}

Cool, let's put in a nice function

In [22]:
function process(input) {
  const rows = input.split('\n');
  return rows.map(row => {
    const parts = row.split(' ');
    
    return {
      state: parts[0].slice(1,-1).split('').map(e => e === "#"),
      buttons: parts.slice(1,-1).map(s => s.slice(1,-1).split(',').map(n => parseInt(n))),
      jolts: parts[parts.length-1].slice(1, -1).split(',').map(n => parseInt(n))
    };
  });
}
process(sample)

[
  {
    state: [ false, true, true, false ],
    buttons: [ [ 3 ], [ 1, 3 ], [ 2 ], [ 2, 3 ], [ 0, 2 ], [ 0, 1 ] ],
    jolts: [ 3, 5, 4, 7 ]
  },
  {
    state: [ false, false, false, true, false ],
    buttons: [ [ 0, 2, 3, 4 ], [ 2, 3 ], [ 0, 4 ], [ 0, 1, 2 ], [ 1, 2, 3, 4 ] ],
    jolts: [ 7, 5, 12, 7, 2 ]
  },
  {
    state: [ false, true, true, true, false, true ],
    buttons: [ [ 0, 1, 2, 3, 4 ], [ 0, 3, 4 ], [ 0, 1, 2, 4, 5 ], [ 1, 2 ] ],
    jolts: [ 10, 11, 11, 5, 10, 5 ]
  }
]

Okay, after some parsing bugs we are on. Let's start building the BFS.

In [23]:
const one = process(sample)[0];
one

{
  state: [ false, true, true, false ],
  buttons: [ [ 3 ], [ 1, 3 ], [ 2 ], [ 2, 3 ], [ 0, 2 ], [ 0, 1 ] ],
  jolts: [ 3, 5, 4, 7 ]
}

Let's apply the button (1,3) to the initial state

In [24]:
[false, false, false, false].map((v, ind) => one.buttons[1].includes(ind) ? !v : v);

[ false, true, false, true ]

In [25]:
const apply = (st, but) => st.map((v, ind) => but.includes(ind) ? !v : v);
apply(apply([false, false, false, false], one.buttons[1]), one.buttons[1])

[ false, false, false, false ]

You know, now that I think about it, I think this problem can be thought of as a modular arithmetic problem. Let me illustrate:

- (3) = 0001
- (1,3) = 0101
- (2) = 0010
- (2,3) = 0011
- (0,2) = 1010
- (0,1) = 1100

We start at zero and we need to reach
- `[.##.]` = 0110

Activating a number is equivalent to... wait,no, I'm wrong, it's not addition, it's XOR. Well, it was an interesting thought. Anyway this does mean we could represent everything with bitarrays and use bitwise XOR if performance requires. Let's get back then. Let's use BFS to find the shortest list of button presses.

In [26]:
const tmp = [1,2,3];

In [27]:
tmp.push(4);
tmp.shift();
tmp

[ 2, 3, 4 ]

In [28]:
const eq = (a, b) => {
  const n = a.length;
  if (b.length != n) throw new Exception("arrays differ in length");
  for (let i = 0; i < n; i++) {
    if (a[i] != b[i]) return false;
  }
  return true;
}

eq([1,2,3], [1,2,4])

false

In [29]:
const p = process(sample)[2];
const initial = p.state.map(v => false);
const q = [[initial, 0]];
const added = new Set([initial]);
while (q.length > 0) {
  const [cur_st, d] = q.shift();
  if (eq(cur_st, p.state)) {
    console.log(cur_st, d);
    break;
  }
  for (const b of p.buttons) {
    const new_st = apply(cur_st, b);
    if (!added.has(new_st)) {
      q.push([new_st, d + 1]);
      added.add(new_st);
    }
  }
}

[ false, true, true, true, false, true ] 2


In [30]:
function shortest_clicks(p) {
  const initial = p.state.map(v => false);
  const q = [[initial, 0]];
  const added = new Set([initial]);
  while (q.length > 0) {
    const [cur_st, d] = q.shift();
    if (eq(cur_st, p.state)) {
      return d;
    }
    for (const b of p.buttons) {
      const new_st = apply(cur_st, b);
      if (!added.has(new_st)) {
        q.push([new_st, d + 1]);
        added.add(new_st);
      }
    }
  }
}
shortest_clicks(process(sample)[1])

3

Okay, looks good, let's wrap it up

In [31]:
function part1(input) {
  const rows = process(input);
  // const results = rows.map(shortest_clicks);
  // return results.reduce((acc, x) => acc + x);
  let sum = 0;
  for (let i = 0; i < rows.length; i++) {
    console.log(i);
    sum += shortest_clicks(rows[i]);
  }
}
// part1(input)

We seem to hang on rows 24 of the input. let's debug.

`[...######] (0,2,7,8) (0,1,3,4,5,6,7) (1,4,5,8) (0,1,2,3,5,6,7) (2) (3,6,8) (1,4,6) (0,1,3,7,8) (5,6,7) (3,4) {28,45,36,35,29,44,50,39,38}
`

In [32]:
const rows=  process(input);
rows[23]

{
  state: [
    false, false, false,
    true,  true,  true,
    true,  true,  true
  ],
  buttons: [
    [ 0, 2, 7, 8 ],
    [
      0, 1, 3, 4,
      5, 6, 7
    ],
    [ 1, 4, 5, 8 ],
    [
      0, 1, 2, 3,
      5, 6, 7
    ],
    [ 2 ],
    [ 3, 6, 8 ],
    [ 1, 4, 6 ],
    [ 0, 1, 3, 7, 8 ],
    [ 5, 6, 7 ],
    [ 3, 4 ]
  ],
  jolts: [
    28, 45, 36, 35, 29,
    44, 50, 39, 38
  ]
}

Parsing looks correct.

In [33]:
const p = rows[23];
const initial = p.state.map(v => false);
const q = [[initial, 0]];
const added = new Set([initial]);
let i = 0; // let's cap the loop
while (q.length > 0 && i < 1) {
  i += 1;
  const [cur_st, d] = q.shift();
  // console.log(cur_st);
  if (eq(cur_st, p.state)) {
    console.log(cur_st, d);
    break;
  }
  for (const b of p.buttons) {
    const new_st = apply(cur_st, b);
    if (!added.has(new_st)) {
      q.push([new_st, d + 1]);
      added.add(new_st);
    }
  }
}

Set(11) {
  [
    false, false,
    false, false,
    false, false,
    false, false,
    false
  ],
  [
    true,  false, true,
    false, false, false,
    false, true,  true
  ],
  [
    true, true, false,
    true, true, true,
    true, true, false
  ],
  [
    false, true,  false,
    false, true,  true,
    false, false, true
  ],
  [
    true, true,  true,
    true, false, true,
    true, true,  false
  ],
  [
    false, false,
    true,  false,
    false, false,
    false, false,
    false
  ],
  [
    false, false, false,
    true,  false, false,
    true,  false, true
  ],
  [
    false, true,  false,
    false, true,  false,
    true,  false, false
  ],
  [
    true,  true,  false,
    true,  false, false,
    false, true,  true
  ],
  [
    false, false, false,
    false, false, true,
    true,  true,  false
  ],
  [
    false, false, false,
    true,  true,  false,
    false, false, false
  ]
}

Why are we getting all that output? anyway, I have a suspicion that set can't handle arrays correctly, let's check.

In [34]:
new Set([[1],[1]])

Set(2) { [ 1 ], [ 1 ] }

yep, as suspected. We can use a hack though

In [35]:
new Set([JSON.stringify([1]),JSON.stringify([1])])

Set(1) { "[1]" }

okay, let's fix that and retry

In [36]:
function shortest_clicks(p) {
  const initial = p.state.map(v => false);
  const q = [[initial, 0]];
  const added = new Set([JSON.stringify(initial)]);
  while (q.length > 0) {
    const [cur_st, d] = q.shift();
    if (eq(cur_st, p.state)) {
      return d;
    }
    for (const b of p.buttons) {
      const new_st = apply(cur_st, b);
      if (!added.has(JSON.stringify(new_st))) {
        q.push([new_st, d + 1]);
        added.add(JSON.stringify(new_st));
      }
    }
  }
}
function part1(input) {
  const rows = process(input);
  const results = rows.map(shortest_clicks);
  return results.reduce((acc, x) => acc + x);
}
part1(input)

390

Success!

## Part 2
Now instead of XORs, we have addition and the target is the jolts. I went for a run after finishing part 1, and realized that the problem can be formulated as a linear equation with an objective function. Each button is a vector over $\mathbb{Z}_2$, and we are looking for the linear combination that yields the target state. The problem is over defined, as in there are more buttons then dimensions. This also means the number of steps is bound by the dimension, so this bounds the depth of the search in the BFS. We could have solved it by trying out different subsets and calculating the linear combination or something.

Anyhow, part 2 stops it from being so easy, because in this lens it becomes an Integer Linear Programming problem. We can keep trying BFS and prune branches that are dead ends (because of the mono-increasing nature of the counters). I suspect we would run into performance issues, but let's give it a try.

In [37]:
const apply2 = (st, but) => st.map((v, ind) => but.includes(ind) ? v+1 : v);
const any_gt = (a, b) => {
  const n = a.length;
  if (b.length != n) throw new Exception("arrays differ in length");
  for (let i = 0; i < n; i++) {
    if (a[i] > b[i]) return true;
  }
  return false;
}

function shortest_clicks2(p) {
  const initial = p.state.map(v => 0);
  const q = [[initial, 0]];
  const added = new Set([JSON.stringify(initial)]);
  while (q.length > 0) {
    const [cur_st, d] = q.shift();
    if (any_gt(cur_st, p.jolts)) {
      continue;
    }
    if (eq(cur_st, p.jolts)) {
      return d;
    }
    for (const b of p.buttons) {
      const new_st = apply2(cur_st, b);
      if (!added.has(JSON.stringify(new_st))) {
        q.push([new_st, d + 1]);
        added.add(JSON.stringify(new_st));
      }
    }
  }
}
function part2(input) {
  const rows = process(input);
  // const results = rows.map(shortest_clicks2);
  // return results.reduce((acc, x) => acc + x);
  let sum = 0;
  for (let i=0; i < rows.length; i++) {
    console.log(i);
    sum += shortest_clicks2(rows[i])
  }
  return sum;
}
part2(sample);

0
1
2


33

In [38]:
// part2(input);

This already hangs on number 0.

Our next option is to try and think of optimizations. I also think this is reminiscent of A*, since we have the heuristic that buttons which increase more counters are more attractive. But I think I want to try a different approach. Usually I try to implement all the code myself, but in this case we can try and use a ILP solver.

In [39]:
import solver from "npm:javascript-lp-solver";

Here's an example from Google AI overview

In [40]:
const model = {
  optimize: 'profit',
  opType: 'max',
  variables: {
    productA: { profit: 10, material: 2, labor: 1 },
    productB: { profit: 15, material: 3, labor: 2 }
  },
  constraints: {
    material: { max: 100 },
    labor: { max: 50 }
  },
  ints: { // Optional: for integer variables
    productA: 1,
    productB: 1
  }
}

const solution = solver.Solve(model);

console.log(solution);

{
  feasible: true,
  result: 500,
  bounded: true,
  isIntegral: true,
  productA: 50
}


What we optimize here are the number of products manufactured from each type. The goal is to maximize the profit, as defined in `optimize`, and we say we want to maximize it. The products are defined in the variables, and for each variable it has a certain profit, material usage and labor usage. The `constraints` cap the material and labor, and `ints` require the variables to be integers.

So how do we model it for our case?
- The variables will be the button presses, so we need a var for each button
- Each button contributes to the relevant positions, and contributes 1 to `total_presses`
- Constraints are that the relevant positions need to add up to the target
- All the variables are ints
- We might need a constraint for non-negativity, we'll see.

Let's start by initializing the variables struct

In [44]:
rows[0].buttons

[
  [ 0, 1, 2, 3, 6 ],
  [ 1, 3 ],
  [ 2, 3 ],
  [ 0, 1, 2, 5, 6, 7 ],
  [ 4, 6, 7 ],
  [ 0, 1, 3, 4, 6, 7 ],
  [ 1, 3, 7 ],
  [ 3, 4, 5 ],
  [ 3, 5 ],
  [ 1 ]
]

In [46]:
Object.fromEntries([[0,1]])

{ "0": 1 }

In [67]:
const get_vars = row => Object.fromEntries(row.buttons.map((b, i) => [`b${i}`, {"total_presses": 1, ...Object.fromEntries(b.map(j => [`pos${j}`, 1]))}]))
get_vars(rows[0])

{
  b0: { total_presses: 1, pos0: 1, pos1: 1, pos2: 1, pos3: 1, pos6: 1 },
  b1: { total_presses: 1, pos1: 1, pos3: 1 },
  b2: { total_presses: 1, pos2: 1, pos3: 1 },
  b3: {
    total_presses: 1,
    pos0: 1,
    pos1: 1,
    pos2: 1,
    pos5: 1,
    pos6: 1,
    pos7: 1
  },
  b4: { total_presses: 1, pos4: 1, pos6: 1, pos7: 1 },
  b5: {
    total_presses: 1,
    pos0: 1,
    pos1: 1,
    pos3: 1,
    pos4: 1,
    pos6: 1,
    pos7: 1
  },
  b6: { total_presses: 1, pos1: 1, pos3: 1, pos7: 1 },
  b7: { total_presses: 1, pos3: 1, pos4: 1, pos5: 1 },
  b8: { total_presses: 1, pos3: 1, pos5: 1 },
  b9: { total_presses: 1, pos1: 1 }
}

In [91]:
const get_cons = row => Object.fromEntries(row.jolts.map((jolt, i) => [`pos${i}`, {equal: jolt}]));
get_cons(rows[0])

{
  pos0: { equal: 22 },
  pos1: { equal: 31 },
  pos2: { equal: 23 },
  pos3: { equal: 55 },
  pos4: { equal: 31 },
  pos5: { equal: 29 },
  pos6: { equal: 35 },
  pos7: { equal: 36 }
}

Ok I think we have everything. Let's try

In [105]:
const rows = process(sample);
const row = rows[0];
const variables = get_vars(row);
const constraints = get_cons(row);
const ints = Object.fromEntries(Object.keys(tmp).map(k=> [k, 1]));
const model = {
  optimize: 'total_presses',
  opType: 'min',
  variables,
  constraints,
  ints
}
console.log(model);

const solution = solver.Solve(model);

console.log(solution);

{
  optimize: "total_presses",
  opType: "min",
  variables: {
    b0: { total_presses: 1, pos3: 1 },
    b1: { total_presses: 1, pos1: 1, pos3: 1 },
    b2: { total_presses: 1, pos2: 1 },
    b3: { total_presses: 1, pos2: 1, pos3: 1 },
    b4: { total_presses: 1, pos0: 1, pos2: 1 },
    b5: { total_presses: 1, pos0: 1, pos1: 1 }
  },
  constraints: {
    pos0: { equal: 3 },
    pos1: { equal: 5 },
    pos2: { equal: 4 },
    pos3: { equal: 7 }
  },
  ints: {
    b0: 1,
    b1: 1,
    b2: 1,
    b3: 1,
    b4: 1,
    b5: 1,
    b6: 1,
    b7: 1,
    b8: 1,
    b9: 1
  }
}
{
  feasible: true,
  result: 10,
  bounded: true,
  isIntegral: true,
  b4: 3,
  b1: 5,
  b3: 1,
  b0: 1
}


Looks correct. Let's sum and wrap it up.

In [122]:
const get_vars = row => Object.fromEntries(
  row.buttons.map((b, i) => 
    [`b${i}`, {
      "total_presses": 1,
      ...Object.fromEntries(
          b.map(j => [`pos${j}`, 1])
      )}
    ]
  )
)

const get_cons = row => Object.fromEntries(row.jolts.map((jolt, i) => [`pos${i}`, {equal: jolt}]));

function solve(row) {
  const variables = get_vars(row);
  const constraints = get_cons(row);
  const ints = Object.fromEntries(Object.keys(variables).map(k=> [k, 1]));
  const model = {
    optimize: 'total_presses',
    opType: 'min',
    variables,
    constraints,
    ints
  }
  
  const solution = solver.Solve(model);
  return {solution, 
          sum: Object.keys(ints).map(k => solution[k] ?? 0).reduce((acc, x) => acc + x)}
}
solve(rows[0])

{
  solution: {
    feasible: true,
    result: 10,
    bounded: true,
    isIntegral: true,
    b4: 3,
    b1: 5,
    b3: 1,
    b0: 1
  },
  sum: 10
}

In [124]:
function part2(input) {
  const rows = process(input);
  const results = rows.map(solve);
  return results.reduce((acc, x) => acc + x.sum, 0);
}
part2(sample)

33

In [125]:
part2(input)

14677

Hmm, not the correct solution. Need to debug... two options I can think of

- the solver doesnt return the correct solution
- the solution is correct but not optimal
- ???

Let's build a verifier for one...

Actually had claude look at the notebook and it spotted a place where I used `tmp` instead of `variables` in `solve`, so that's that.

## Retrospective

I broke the no external dependencies rule, but sometimes you got to decompose problems. It is an invitation to learn how to implement linear programming algorithms though, might do that sometimes...

Asked for feedback from some AI models, Gemini gave an implementation with bitmasks, lets check it out.

In [128]:
function solvePart1Bitmask(p) {
  // Convert target state array to a single integer
  // e.g., [F, T, T, F] -> 0110 -> 6
  let targetMask = 0;
  p.state.forEach((isOn, idx) => {
    if (isOn) targetMask |= (1 << idx);
  });

  // Convert buttons to masks
  const buttonMasks = p.buttons.map(btnIndices => 
    btnIndices.reduce((mask, idx) => mask | (1 << idx), 0)
  );

  const initial = 0;
  const q = [[initial, 0]];
  const visited = new Set([initial]); // Set<number> is very fast

  while (q.length > 0) {
    const [currentMask, dist] = q.shift();

    if (currentMask === targetMask) return dist;

    for (const bMask of buttonMasks) {
      // XOR toggles the bits
      const nextMask = currentMask ^ bMask;
      
      if (!visited.has(nextMask)) {
        visited.add(nextMask);
        q.push([nextMask, dist + 1]);
      }
    }
  }
}

solvePart1Bitmask(rows[2])

2

In [132]:
// So we use integers as bitarrays
// | is bitwise OR
1 | 2

3

In [136]:
// << does shift left
[1 << 0, 1 << 1, 1 << 2, 1 << 3]

[ 1, 2, 4, 8 ]

Then we generate the masks for the buttons and for the target state, and ^ is like pressing the button. And since it's just a number, we can throw it in a set. That's dope.